# 🚀 DENSO VisionMind — Dual AI Serving: Qwen2.5-VL & ColPali Engine (Kaggle 2 x GPU T4)

Notebook này triển khai **HỆ THỐNG AI ĐA PHƯƠNG THỨC KÉP** trên **2 GPU T4** của Kaggle:
- 🧠 **GPU 0 (`cuda:0`)**: **vLLM Engine** phục vụ **Qwen2.5-VL-7B-Instruct-AWQ** (OpenAI-compatible API trên cổng nội bộ 8002).
- 👁️ **GPU 1 (`cuda:1`)**: **ColPali Engine** (`vidore/colpali-v1.2` trên nền PaliGemma-3B bfloat16) phục vụ **Late Interaction Multi-Vector MaxSim Search** từ pixel bản vẽ CAD & sơ đồ PDF.
- 🌐 **FastAPI Gateway (Cổng 8000)**: Tổng hợp cả 2 dịch vụ (`/v1/...` và `/colpali/...`) và expose qua 1 đường hầm **ngrok** duy nhất.

### ⚠️ YÊU CẦU BẮT BUỘC TRƯỚC KHI CHẠY:
Ở bảng **Settings** góc phải màn hình Kaggle:
1. **Accelerator**: Chọn **GPU T4 x 2**.
2. **Internet**: Bật **ON**.

In [ ]:
# Cell 1: Kiểm tra phần cứng 2 GPU T4 trên Kaggle
import torch

print("🔍 Đang kiểm tra GPU...")
if not torch.cuda.is_available() or torch.cuda.device_count() < 2:
    print(f"⚠️ Cảnh báo: Số GPU hiện có là {torch.cuda.device_count() if torch.cuda.is_available() else 0}.")
    print("👉 Hãy nhìn sang góc phải màn hình Kaggle: Settings -> Accelerator -> Chọn 'GPU T4 x 2' để có đủ 2 GPU!")
else:
    print(f"✅ Đã sẵn sàng 2 GPU: GPU 0 = {torch.cuda.get_device_name(0)} | GPU 1 = {torch.cuda.get_device_name(1)}")
!nvidia-smi --query-gpu=index,name,memory.total --format=csv

In [ ]:
# Cell 2: Cài đặt các thư viện vLLM, ColPali, FastAPI, uvicorn và pyngrok
# Gỡ bỏ torchao cũ (0.10.0) trên Kaggle để tránh xung đột với peft khi nạp ColPali
!pip uninstall -y torchao
!pip install -q -U vllm pyngrok colpali-engine fastapi uvicorn httpx python-multipart
print("✅ Cài đặt môi trường thành công!")


In [ ]:
# Cell 3: Cấu hình Hugging Face Token & ngrok Authtoken
import os
from pyngrok import ngrok

HF_TOKEN = os.getenv("HF_TOKEN", "YOUR_HF_TOKEN_HERE")
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN

NGROK_AUTH_TOKEN = os.getenv("NGROK_AUTH_TOKEN", "YOUR_NGROK_AUTH_TOKEN_HERE")
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

print("✅ Đã cấu hình xác thực thành công cả HuggingFace và ngrok!")

In [ ]:
%%writefile gateway_server.py
# ==============================================================================
# GATEWAY SERVER: TỔNG HỢP VLLM (GPU 0) VÀ COLPALI VLM (GPU 1)
# File này được tự động tạo bởi lệnh %%writefile của Jupyter
# ==============================================================================
import os
import io
import base64
from PIL import Image
import torch
from fastapi import FastAPI, Request, Response
from pydantic import BaseModel
import httpx

app = FastAPI(title="DENSO Dual VisionMind Gateway: Qwen2.5-VL & ColPali v1.2")

# 1. Khởi tạo mô hình ColPali v1.2 độc lập trên GPU 1 (16GB VRAM)
device_colpali = "cuda:1" if torch.cuda.device_count() > 1 else "cuda:0"
print(f"👁️ [ColPali] Đang nạp mô hình ColPali v1.2 (PaliGemma-3B) lên {device_colpali}...")

from colpali_engine.models import ColPali, ColPaliProcessor

colpali_model = ColPali.from_pretrained(
    "vidore/colpali-v1.2",
    torch_dtype=torch.bfloat16,
    device_map=device_colpali
)
colpali_processor = ColPaliProcessor.from_pretrained("vidore/colpali-v1.2")
colpali_model.eval()
print(f"✅ [ColPali] Đã nạp thành công ColPali v1.2 trên {device_colpali}!")

# 2. Endpoints cho ColPali
class EmbedPageRequest(BaseModel):
    image_base64: str

class EmbedQueryRequest(BaseModel):
    query: str

@app.post("/colpali/embed_page")
def embed_page(req: EmbedPageRequest):
    raw = base64.b64decode(req.image_base64)
    img = Image.open(io.BytesIO(raw)).convert("RGB")
    batch = colpali_processor.process_images([img]).to(colpali_model.device)
    with torch.no_grad():
        emb = colpali_model(**batch)
    vecs = emb.squeeze(0).cpu().to(torch.float32).tolist()
    return {"status": "success", "vector_dim": 128, "num_tokens": len(vecs), "embeddings": vecs}

@app.post("/colpali/embed_query")
def embed_query(req: EmbedQueryRequest):
    batch = colpali_processor.process_queries([req.query]).to(colpali_model.device)
    with torch.no_grad():
        emb = colpali_model(**batch)
    vecs = emb.squeeze(0).cpu().to(torch.float32).tolist()
    return {"status": "success", "vector_dim": 128, "num_tokens": len(vecs), "embeddings": vecs}

@app.get("/colpali/health")
def health():
    return {"status": "ready", "engine": "ColPali v1.2 (PaliGemma-3B)", "device": str(colpali_model.device)}

# 3. Reverse Proxy trong suốt chuyển hướng /v1/... tới vLLM Qwen2.5-VL trên GPU 0 (cổng 8002)
vllm_client = httpx.AsyncClient(base_url="http://127.0.0.1:8002", timeout=180.0)

@app.api_route("/v1/{path:path}", methods=["GET", "POST", "PUT", "DELETE"])
async def vllm_proxy(path: str, request: Request):
    url = f"/v1/{path}"
    headers = dict(request.headers)
    headers.pop("host", None)
    body = await request.body()
    response = await vllm_client.request(request.method, url, headers=headers, content=body, params=request.query_params)
    return Response(content=response.content, status_code=response.status_code, headers=dict(response.headers))


In [ ]:
# Cell 5: Khởi động song song 2 AI Engine & Mở cổng ngrok Tunnel
import os
import subprocess
import time
import gc
import torch
import requests
from pyngrok import ngrok

# 1. Dọn dẹp tiến trình cũ & giải phóng triệt để VRAM trên cả 2 GPU
print("🧹 Đang giải phóng cổng mạng và VRAM trên cả 2 GPU...")
current_pid = os.getpid()
for p in subprocess.getoutput("nvidia-smi --query-compute-apps=pid --format=csv,noheader").split():
    if p.strip().isdigit() and int(p.strip()) != current_pid:
        try:
            os.kill(int(p.strip()), 9)
        except Exception:
            pass

!fuser -k 8000/tcp || true
!fuser -k 8002/tcp || true
!pkill -9 -f "vllm" || true
!pkill -9 -f "uvicorn" || true
time.sleep(2)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# 2. Khởi động vLLM Server chạy trên GPU 0 (cổng nội bộ 8002)
print("🚀 [GPU 0] Khởi động vLLM Qwen2.5-VL-7B-Instruct-AWQ trên cổng 8002...")
vllm_cmd = (
    "CUDA_VISIBLE_DEVICES=0 vllm serve Qwen/Qwen2.5-VL-7B-Instruct-AWQ "
    "--port 8002 "
    "--gpu-memory-utilization 0.80 "
    "--max-model-len 4096 "
    "--trust-remote-code "
    "--dtype half"
)
vllm_proc = subprocess.Popen(vllm_cmd, shell=True)

# 3. Khởi động Gateway FastAPI chạy ColPali trên GPU 1 (cổng 8000)
print("👁️ [GPU 1] Khởi động Gateway ColPali v1.2 trên cổng 8000...")
gateway_cmd = "uvicorn gateway_server:app --host 0.0.0.0 --port 8000"
gateway_proc = subprocess.Popen(gateway_cmd, shell=True)

# 4. Đợi cả 2 mô hình khởi động xong vào 2 GPU
print("⏳ Đang đợi cả 2 mô hình nạp vào VRAM (thường mất khoảng 60-90 giây)...")
vllm_ready = False
colpali_ready = False

for attempt in range(40):
    time.sleep(5)
    if not colpali_ready:
        try:
            r = requests.get("http://127.0.0.1:8000/colpali/health", timeout=2)
            if r.status_code == 200:
                colpali_ready = True
                print("  ✅ [GPU 1] ColPali Engine đã sẵn sàng!")
        except Exception:
            pass

    if not vllm_ready:
        try:
            r = requests.get("http://127.0.0.1:8002/v1/models", timeout=2)
            if r.status_code == 200:
                vllm_ready = True
                print("  ✅ [GPU 0] vLLM Qwen2.5-VL đã sẵn sàng!")
        except Exception:
            pass

    if vllm_ready and colpali_ready:
        break

if not (vllm_ready and colpali_ready):
    print("⚠️ Lưu ý: Một trong hai engine chưa phản hồi sau thời gian chờ. Đang tiếp tục mở tunnel...")

# 5. Mở ngrok Tunnel cho Gateway duy nhất (cổng 8000)
print("🌐 Đang mở đường hầm ngrok công khai qua Gateway (cổng 8000)...")
ngrok.kill()
tunnel = ngrok.connect(8000)
public_url = tunnel.public_url

print("\n" + "="*70)
print("🎉 ĐÃ TRIỂN KHAI THÀNH CÔNG HỆ THỐNG DUAL VISIONMIND!")
print(f"🔗 Public Gateway URL: {public_url}")
print(f"👉 VLLM_BASE_URL:     {public_url}/v1")
print(f"👉 COLPALI_BASE_URL:  {public_url}/colpali")
print("="*70)


In [ ]:
# Cell 6: Kiểm tra thử nghiệm truy vấn trực tiếp trên Kaggle
import requests

print("🧪 Kiểm tra ColPali:")
try:
    r_col = requests.get("http://127.0.0.1:8000/colpali/health", timeout=3)
    print("ColPali status:", r_col.json())
except Exception as e:
    print("ColPali check err:", e)

print("\n🧪 Kiểm tra vLLM Models:")
try:
    r_vllm = requests.get("http://127.0.0.1:8000/v1/models", timeout=3)
    print("vLLM models:", r_vllm.json())
except Exception as e:
    print("vLLM check err:", e)